# Colon Xenium slide QC summary

## Goal

Combine exactly six validated region bundles from one run label and execution mode. The summary compares technical QC across Mouse_1 and Mouse_2 at top, middle, and bottom positions without treating cells as biological replicates.

## Setup

HPC paths are defaults. Local validation injects D-drive paths and `LOCAL_SUBSET`. No package installation or writes outside `colon_analysis` are permitted.

In [ ]:
EXECUTION_MODE <- toupper(Sys.getenv("COLON_QC_MODE", "FULL_HPC"))
PROJECT_ROOT <- Sys.getenv("COLON_PROJECT_ROOT", "/dssg/home/acct-svetoslav_chakarov/svetoslav_chakarov/Lab_members/Yanan_Hu/YNH_Xenium")
PIPELINE_REPO <- Sys.getenv("COLON_PIPELINE_REPO", file.path(PROJECT_ROOT, "colon_analysis", "YNH_Xenium_Colon"))
RUN_LABEL <- Sys.getenv("COLON_RUN_LABEL", "colon_qc_hpc")
RUN_ROOT <- Sys.getenv("COLON_RUN_ROOT", file.path(PROJECT_ROOT, "colon_analysis", "colon_qc_outputs", RUN_LABEL))
TEMP_ROOT <- Sys.getenv("COLON_TEMP_ROOT", file.path(PROJECT_ROOT, "colon_analysis", "tmp", RUN_LABEL))
dir.create(TEMP_ROOT, recursive = TRUE, showWarnings = FALSE)
Sys.setenv(TMPDIR = TEMP_ROOT, TMP = TEMP_ROOT, TEMP = TEMP_ROOT)
source(file.path(PIPELINE_REPO, "R", "source.R"))
regions <- expected_colon_regions()

## Inputs and integrity

Require Region_1 through Region_6, reject missing/duplicate regions and mixed run labels or modes, then reload every saved sparse object and table.

In [ ]:
manifest <- utils::read.delim(file.path(PIPELINE_REPO, "config", "colon_sample_manifest.tsv"), check.names = FALSE)
validate_sample_manifest(manifest, regions)
slide_data <- read_colon_slide_qc_outputs(RUN_ROOT, expected_regions = regions, run_label = RUN_LABEL, execution_mode = EXECUTION_MODE)
stopifnot(identical(slide_data$coverage$region_id, regions))

## Cell QC

Summarize fixed-bound pass fractions and review flags. `primary_include` is never redefined at slide level. Region readiness is derived from current evidence rather than inherited anchor or sensitivity labels.

In [ ]:
slide_summary <- summarise_slide_qc(slide_data)
position_summary <- summarise_colon_mouse_position(slide_summary$section_summary, manifest)
position_summary$within_mouse
position_summary$matched_position

## Outputs and checks

Save combined tables, descriptive within-mouse and matched-position summaries, readiness gates, figures, session information, and a reloadable slide object.

In [ ]:
summary_output_dir <- file.path(RUN_ROOT, "slide_summary")
result <- write_colon_slide_qc_bundle(PROJECT_ROOT, summary_output_dir, RUN_LABEL, EXECUTION_MODE, slide_data, slide_summary, position_summary)
stopifnot(validate_colon_slide_qc_bundle(summary_output_dir, expected_regions = regions))
result

## Next steps

Use full-HPC evidence for the final QC decision. With only two mice, anatomical-position results are descriptive. The adipose-derived eosinophil lists remain hypotheses (`AT-short-like` and `AT-long-like`) until colon-specific coherence and identity checks are completed in a later, study-specific analysis stage.